<a href="https://colab.research.google.com/github/ashikjoel/-Context-Aware-Neural-Recommendation-Engine/blob/ashik_joel/Context_Aware_Neural_Recommendation_Engine(week_2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [6]:
!pip install -q tensorflow-recommenders tf-keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 10.5 MB/s eta 0:00:00


In [7]:
import tensorflow as tf
import tensorflow_recommenders as tfrs

print("TensorFlow:", tf.__version__)
print("TFRS:", tfrs.__version__)

TensorFlow: 2.20.0
TFRS: v0.7.7


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HM-Recommendation-System")
    .getOrCreate()
)

In [10]:
PROCESSED_PATH = "/content/drive/MyDrive/Recommendation_Engine/data/processed"
FEATURE_PATH = "/content/drive/MyDrive/Recommendation_Engine/features"
VOCAB_PATH = "/content/drive/MyDrive/Recommendation_Engine/vocabularies"

In [11]:
customers_df = spark.read.parquet(f"{PROCESSED_PATH}/customers_clean.parquet")

articles_df = spark.read.parquet(f"{PROCESSED_PATH}/articles_clean.parquet")

transactions_df = spark.read.parquet(f"{PROCESSED_PATH}/transactions_clean.parquet")

In [12]:
recency_df = spark.read.parquet(f"{FEATURE_PATH}/recency.parquet")

product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/product_popularity.parquet"
)

monthly_product_popularity_df = spark.read.parquet(
    f"{FEATURE_PATH}/monthly_product_popularity.parquet"
)

In [13]:
customer_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/customer_vocab.parquet"
)

article_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/article_vocab.parquet"
)

product_type_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/product_type_vocab.parquet"
)

department_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/department_vocab.parquet"
)

color_vocab = spark.read.parquet(
    f"{VOCAB_PATH}/color_vocab.parquet"
)

In [14]:
print("Processed Datasets")
print("------------------")
print("Customers      :", customers_df.count())
print("Articles       :", articles_df.count())
print("Transactions   :", transactions_df.count())

print("\nFeature Tables")
print("------------------")
print("Recency                :", recency_df.count())
print("Product Popularity     :", product_popularity_df.count())
print("Monthly Popularity     :", monthly_product_popularity_df.count())

print("\nVocabularies")
print("------------------")
print("Customer Vocabulary    :", customer_vocab.count())
print("Article Vocabulary     :", article_vocab.count())
print("Product Type Vocabulary:", product_type_vocab.count())
print("Department Vocabulary  :", department_vocab.count())
print("Color Vocabulary       :", color_vocab.count())

Processed Datasets
------------------
Customers      : 1371980
Articles       : 105542
Transactions   : 31788324

Feature Tables
------------------
Recency                : 1362281
Product Popularity     : 104547
Monthly Popularity     : 768883

Vocabularies
------------------
Customer Vocabulary    : 1371980
Article Vocabulary     : 105542
Product Type Vocabulary: 131
Department Vocabulary  : 250
Color Vocabulary       : 50


#Create the Interaction Dataset

In [15]:
interactions_df = transactions_df.select(
    "customer_id",
    "article_id"
)

print("Total Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Total Interactions: 31788324
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016003 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016001 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|682236013 |
|aaa78c87aacba903d16f393da3edeca27d62e642b1a639a882f4dac368626041|706016016 |
|aaa7a0483dd5b9e395d95324dcbfeb617af9800f39487d4b6aaee662bcd384c7|783335003 |
|aaa7b371465a823fec4312ef0f2807f924d54e5d41afb686223b76266bd9c599|563519008 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|783056001 |
|aaa85d7e704f11e59526f493b9d6016d0d6540a0aa131a4aeec8b76e7c775aa2|695325016 |
|aaa8f491632b9022bf20aa444c793bdf23621bd9463c050e86b73ada4cb059b6|757333001 |
|aaa8f491632b9022bf20aa444c793bdf23

#Remove Duplicate User–Item Pairs

In [16]:
interactions_df = interactions_df.dropDuplicates()

print("Unique User-Item Interactions:", interactions_df.count())

interactions_df.show(10, truncate=False)

Unique User-Item Interactions: 27306439
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|aaac535f79b71437632d6001ebd960766da3000d7c455cd7a0500dab24cfd50e|799507001 |
|abd2537848661862039af47c8bcee2f9620c5fdc3b13c1af1f66f1f485c51de7|399087021 |
|ac2a5d7aa83653f77dbb343a90ebb705fb3c0a1e2683dbf74b1e08dab042c9bd|821152001 |
|ac43e628b476ec53ee48233d5ff7ad26d91b114095b51c8bdf8f5b560d101218|835730001 |
|acd03ec982613dcc026b69a4323f1db291cfbcd6a20173fab14c1063b7c014f2|708473003 |
|acfcd9df9a2a130cc54f547ea5f5829c5ff1913c9fda8e5b29a7badcbf544e26|737222004 |
|ad410adca76cb24d968194c0c2cf02d4714c2b8a639c9377c61020dc1972e8ef|734623002 |
|adb4d1ca1ae86f0a4592ba7ee5d586662945a45bb8d5a76761d971538f2c7980|728703008 |
|adceb8b35d5250062e3bd8b2a5025ee782874c798227c143ef6691488c75fb4d|399223001 |
|adf5b91a4a8092d8f2ce64e

#Create a Training Sample

In [17]:
from pyspark.sql.functions import rand

training_sample = (
    interactions_df
    .orderBy(rand())
    .sample(withReplacement=False, fraction=0.05, seed=42)
)

print("Training Sample Size:", training_sample.count())

training_sample.show(10, truncate=False)

Training Sample Size: 1366023
+----------------------------------------------------------------+----------+
|customer_id                                                     |article_id|
+----------------------------------------------------------------+----------+
|16eeb3dd325039e20e24b844b85e1417ff49afa8050df9d1c524743d6ac46d55|228257003 |
|aff76c9d4a24e295b98255d365dadb308bb7628c52e039633686e0f9eb4109e9|737489004 |
|994d4c5133936c7697a4072043e9bd2e097a463a831979bedb601dc601678a86|550004002 |
|dff21b7d59d1ae1ab3562da6c6cbab792cb386bce452233ee328f32251c0c626|868762003 |
|022a62b8a1b46793963184482bd4ce37bc444b7e0167ae4d5335c52f4ab3a260|695545007 |
|41f15530a0f1d291da812a9e32161c73159657dfd9a944116fd124938465472e|537688015 |
|c118b04db3a7744e670e529dc20b28fd38ee169b53908e80f60159612ea68d25|675271002 |
|15d3997c14d66453620ee7c469e5800e74f2246acec987bf54e112a607fc2f34|747913002 |
|dbd28a9e37ada6b25caeb14d8ea1aba6a9fab0b394a66741dbd254096c81ebd2|599580076 |
|65a7aab75028a26d53fc9d4948ce3c125

#Convert to NumPy

In [18]:
training_pd = training_sample.toPandas()

customer_ids = training_pd["customer_id"].astype(str).values
article_ids = training_pd["article_id"].astype(str).values

print(customer_ids[:5])
print(article_ids[:5])

['16eeb3dd325039e20e24b844b85e1417ff49afa8050df9d1c524743d6ac46d55'
 'aff76c9d4a24e295b98255d365dadb308bb7628c52e039633686e0f9eb4109e9'
 '994d4c5133936c7697a4072043e9bd2e097a463a831979bedb601dc601678a86'
 'dff21b7d59d1ae1ab3562da6c6cbab792cb386bce452233ee328f32251c0c626'
 '022a62b8a1b46793963184482bd4ce37bc444b7e0167ae4d5335c52f4ab3a260']
['228257003' '737489004' '550004002' '868762003' '695545007']


#Build the TensorFlow Dataset

In [19]:
import tensorflow as tf

interactions_ds = tf.data.Dataset.from_tensor_slices({
    "customer_id": customer_ids,
    "article_id": article_ids
})

In [20]:
for sample in interactions_ds.take(5):
    print(sample)

{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'16eeb3dd325039e20e24b844b85e1417ff49afa8050df9d1c524743d6ac46d55'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'228257003'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'aff76c9d4a24e295b98255d365dadb308bb7628c52e039633686e0f9eb4109e9'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'737489004'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'994d4c5133936c7697a4072043e9bd2e097a463a831979bedb601dc601678a86'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'550004002'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'dff21b7d59d1ae1ab3562da6c6cbab792cb386bce452233ee328f32251c0c626'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'868762003'>}
{'customer_id': <tf.Tensor: shape=(), dtype=string, numpy=b'022a62b8a1b46793963184482bd4ce37bc444b7e0167ae4d5335c52f4ab3a260'>, 'article_id': <tf.Tensor: shape=(), dtype=string, numpy=b'695545007'>}


#Prepare for Training

In [21]:
BATCH_SIZE = 8192

train_ds = (
    interactions_ds
    .shuffle(100_000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

#Create the Lookup Layers

In [22]:
customer_ids_vocab = (
    customer_vocab
    .select("customer_id")
    .toPandas()["customer_id"]
    .astype(str)
    .tolist()
)

article_ids_vocab = (
    article_vocab
    .select("article_id")
    .toPandas()["article_id"]
    .astype(str)
    .tolist()
)

print("Customers:", len(customer_ids_vocab))
print("Articles :", len(article_ids_vocab))

Customers: 1371980
Articles : 105542


#Build the Lookup Layers

In [23]:
customer_lookup = tf.keras.layers.StringLookup(
    vocabulary=customer_ids_vocab,
    mask_token=None
)

article_lookup = tf.keras.layers.StringLookup(
    vocabulary=article_ids_vocab,
    mask_token=None
)

#Test the Lookup

In [24]:
sample_customer = customer_ids[0]
sample_article = article_ids[0]

print("Customer Index:", customer_lookup(tf.constant(sample_customer)).numpy())
print("Article Index :", article_lookup(tf.constant(sample_article)).numpy())

Customer Index: 811924
Article Index : 370


#Build the Query Tower

In [25]:
query_tower = tf.keras.Sequential([
    customer_lookup,

    tf.keras.layers.Embedding(
        input_dim=customer_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

#Test the Query Tower

In [26]:
sample_embedding = query_tower(
    tf.constant([customer_ids[0]])
)

print(sample_embedding.shape)
print(sample_embedding.numpy())

(1, 64)
[[ 0.00619295  0.02517706  0.00705016  0.0007846  -0.02500141 -0.00912223
   0.0080958   0.00889974  0.01469184 -0.00272035 -0.01360379  0.01311319
  -0.02181586 -0.0013007  -0.01490327  0.020605    0.00052354 -0.00521713
   0.00502159 -0.02217673 -0.00240833  0.0020036   0.00043048  0.00710601
  -0.00279327  0.00998212  0.00830269  0.01768777 -0.00852905 -0.01696247
   0.01274339  0.00655642 -0.00126213  0.02095703  0.00936652  0.014702
   0.01243888 -0.00529846  0.01097577  0.00139627  0.01027035  0.018929
  -0.02106396 -0.01051885  0.02158835 -0.01296031  0.00557064  0.00761321
  -0.00529151 -0.00913698  0.00293233 -0.02531191 -0.01616248  0.0149062
   0.01205173 -0.01153108  0.00272096 -0.00548058  0.00409173  0.02335212
  -0.01649374  0.00859884  0.02259189  0.02629626]]


#Build the Candidate Tower

In [27]:
candidate_tower = tf.keras.Sequential([
    article_lookup,

    tf.keras.layers.Embedding(
        input_dim=article_lookup.vocabulary_size(),
        output_dim=64
    ),

    tf.keras.layers.Dense(128, activation="relu"),

    tf.keras.layers.Dense(64)
])

#Test the Candidate Tower

In [28]:
sample_item_embedding = candidate_tower(
    tf.constant([article_ids[0]])
)

print(sample_item_embedding.shape)
print(sample_item_embedding.numpy())

(1, 64)
[[ 0.03347539 -0.00876541  0.01329092  0.02590171  0.00019632  0.01846611
  -0.0178027  -0.00149667  0.02166315 -0.02387004  0.00382789  0.01221051
   0.0274158  -0.01879006  0.00801084 -0.02972101 -0.00331683 -0.01047248
  -0.02825334 -0.01328362 -0.03546335 -0.01241984 -0.01686673  0.0272048
   0.03462726  0.01192058 -0.03111129 -0.03021788 -0.00421144 -0.01770818
  -0.03842676  0.01001074  0.02769475 -0.01651054  0.04840821  0.01669164
  -0.01468988  0.01998808  0.02642483 -0.00584051  0.00893953 -0.01225319
   0.01767849  0.00300398  0.01238423 -0.03227457 -0.00511078 -0.01678156
  -0.02958681  0.01025496 -0.04864024 -0.01801104 -0.00525611 -0.04087506
   0.03563002 -0.01574199 -0.02053899  0.02380704 -0.05136455 -0.03021096
   0.01515293 -0.00640408 -0.00083572  0.01848451]]


#Verify Both Towers

In [29]:
print("User Embedding Shape :", sample_embedding.shape)
print("Item Embedding Shape :", sample_item_embedding.shape)

User Embedding Shape : (1, 64)
Item Embedding Shape : (1, 64)


#Create the Candidate Dataset

In [30]:
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(article_ids_vocab)
    .batch(1024)
)

In [31]:
for batch in candidate_dataset.take(1):
    print(batch[:5])

tf.Tensor([b'108775015' b'108775044' b'108775051' b'110065001' b'110065002'], shape=(5,), dtype=string)


#Build the Retrieval Task

In [32]:
import tensorflow_recommenders as tfrs

retrieval_task = tfrs.tasks.Retrieval(
    metrics=tfrs.metrics.FactorizedTopK(
        candidates=candidate_dataset.map(candidate_tower)
    )
)

#Build the Complete Recommendation Model

In [33]:
class HMRecommendationModel(tfrs.models.Model):

    def __init__(self, query_model, candidate_model):
        super().__init__()

        self.query_model = query_model
        self.candidate_model = candidate_model

        self.task = retrieval_task

    def compute_loss(self, features, training=False):

        user_embeddings = self.query_model(features["customer_id"])

        item_embeddings = self.candidate_model(features["article_id"])

        return self.task(
            user_embeddings,
            item_embeddings
        )

In [34]:
import os

MODEL_PATH = "/content/drive/MyDrive/Recommendation_Engine/models"

os.makedirs(MODEL_PATH, exist_ok=True)

print("Model folder created.")

Model folder created.


In [35]:
query_tower.save(f"{MODEL_PATH}/query_tower.keras")
candidate_tower.save(f"{MODEL_PATH}/candidate_tower.keras")

print("Model architectures saved.")

Model architectures saved.


In [36]:
import json

config = {
    "embedding_dimension": 64,
    "hidden_layer": 128,
    "batch_size": 8192,
    "sample_fraction": 0.05,
    "tensorflow_version": "2.20.0",
    "tfrs_version": "0.7.7"
}

with open(f"{MODEL_PATH}/model_config.json", "w") as f:
    json.dump(config, f, indent=4)

print("Configuration saved.")

Configuration saved.


#Instantiate the Model

In [37]:
model = HMRecommendationModel(
    query_model=query_tower,
    candidate_model=candidate_tower
)

print(model)

#Prepare the Dataset

In [38]:
dataset_size = len(customer_ids)

train_size = int(0.8 * dataset_size)

train_ds = interactions_ds.take(train_size)
test_ds = interactions_ds.skip(train_size)

#Batch the Data

In [39]:
BATCH_SIZE = 8192

train_ds = (
    train_ds
    .shuffle(100_000, seed=42)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    test_ds
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

#Verify the Split

In [40]:
print("Dataset Size :", dataset_size)
print("Training Size:", train_size)
print("Testing Size :", dataset_size - train_size)

Dataset Size : 1364235
Training Size: 1091388
Testing Size : 272847


#Compile the Model

In [41]:
model.compile(
    optimizer=tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

#Train the Model

In [42]:
import os

In [43]:
MODEL_DIR = "/content/drive/MyDrive/Recommendation_Engine/models"

os.makedirs(MODEL_DIR, exist_ok=True)

In [44]:
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(
        MODEL_DIR,
        "weights_epoch_{epoch:02d}.weights.h5"
    ),
    save_weights_only=True,
    save_freq="epoch",
    verbose=1
)

In [45]:
model = HMRecommendationModel(
    query_model=query_tower,
    candidate_model=candidate_tower
)

model.compile(
    optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1)
)

model.load_weights(
    "/content/drive/MyDrive/Recommendation_Engine/models/model_epoch_03.keras"
)

In [46]:
history = model.fit(
    train_ds,
    initial_epoch=3,
    epochs=5,
    callbacks=[checkpoint_callback],
    verbose=1
)

Epoch 4/5
134/134 [==============================] - ETA: 0s - factorized_top_k/top_1_categorical_accuracy: 0.0040 - factorized_top_k/top_5_categorical_accuracy: 0.0047 - factorized_top_k/top_10_categorical_accuracy: 0.0053 - factorized_top_k/top_50_categorical_accuracy: 0.0070 - factorized_top_k/top_100_categorical_accuracy: 0.0083 - loss: 80836.1409 - regularization_loss: 0.0000e+00 - total_loss: 80836.1409 
Epoch 4: saving model to /content/drive/MyDrive/Recommendation_Engine/models/weights_epoch_04.weights.h5


134/134 [==============================] - 4358s 32s/step - factorized_top_k/top_1_categorical_accuracy: 0.0040 - factorized_top_k/top_5_categorical_accuracy: 0.0047 - factorized_top_k/top_10_categorical_accuracy: 0.0053 - factorized_top_k/top_50_categorical_accuracy: 0.0070 - factorized_top_k/top_100_categorical_accuracy: 0.0083 - loss: 80340.6847 - regularization_loss: 0.0000e+00 - total_loss: 80340.6847
Epoch 5/5
134/134 [==============================] - ETA: 0s - factorized_top_k/top_1_categorical_accuracy: 1.4660e-04 - factorized_top_k/top_5_categorical_accuracy: 8.5029e-04 - factorized_top_k/top_10_categorical_accuracy: 0.0016 - factorized_top_k/top_50_categorical_accuracy: 0.0069 - factorized_top_k/top_100_categorical_accuracy: 0.0125 - loss: 71536.7822 - regularization_loss: 0.0000e+00 - total_loss: 71536.7822 
Epoch 5: saving model to /content/drive/MyDrive/Recommendation_Engine/models/weights_epoch_05.weights.h5


134/134 [==============================] - 4400s 33s/step - factorized_top_k/top_1_categorical_accuracy: 1.4660e-04 - factorized_top_k/top_5_categorical_accuracy: 8.5029e-04 - factorized_top_k/top_10_categorical_accuracy: 0.0016 - factorized_top_k/top_50_categorical_accuracy: 0.0069 - factorized_top_k/top_100_categorical_accuracy: 0.0125 - loss: 71107.4415 - regularization_loss: 0.0000e+00 - total_loss: 71107.4415
